In [1]:
# Install the necessary packages
!pip install sounddevice soundfile librosa numpy ipywidgets scipy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!jupyter nbextension enable --py widgetsnbextension

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: kernel kernelspec migrate run troubleshoot

Jupyter command `jupyter-nbextension` not found.


In [2]:
import os
import time
import threading
import numpy as np
import sounddevice as sd
import soundfile as sf
import librosa
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output

# Define your classes
SENTENCES = [
    'Ma_Khusi_Xu',
    'Malai_Mero_Desh_Pyaro_Lagxa',
    'Namaste',
    'Ram_Le_Vaat_Khanxa',
    'Tapaiko_Ghar_Kaha_Xa',
    'TimiLai_Kasto_Chha',
    'Uh_Mero_Mitra_Ho',
    'Yo_Hamro_AI_Ko_Project_ho'
]

BASE_DIR = os.path.join('data', 'raw')

# Create directories if they don't exist
for sentence in SENTENCES:
    os.makedirs(os.path.join(BASE_DIR, sentence), exist_ok=True)

print(f"✅ Folder structure successfully created in '{BASE_DIR}'.")

✅ Folder structure successfully created in 'data/raw'.


In [3]:
def augment_audio(audio_data, sr):
    """Generates 10 augmented variations of the original audio."""
    augmentations = []
    
    # Ensure audio is 1D (mono) for librosa processing
    if audio_data.ndim > 1:
        audio_data = audio_data.flatten()
        
    # 1. Add light noise
    noise_low = audio_data + 0.005 * np.random.randn(len(audio_data))
    augmentations.append(('noise_low', noise_low))
    
    # 2. Add heavy noise
    noise_high = audio_data + 0.01 * np.random.randn(len(audio_data))
    augmentations.append(('noise_high', noise_high))
    
    # 3. Pitch shift up
    pitch_up = librosa.effects.pitch_shift(y=audio_data, sr=sr, n_steps=2)
    augmentations.append(('pitch_up', pitch_up))
    
    # 4. Pitch shift down
    pitch_down = librosa.effects.pitch_shift(y=audio_data, sr=sr, n_steps=-2)
    augmentations.append(('pitch_down', pitch_down))
    
    # 5. Speed up (1.2x)
    speed_up = librosa.effects.time_stretch(y=audio_data, rate=1.2)
    augmentations.append(('speed_up', speed_up))
    
    # 6. Speed down (0.8x)
    speed_down = librosa.effects.time_stretch(y=audio_data, rate=0.8)
    augmentations.append(('speed_down', speed_down))
    
    # 7. Volume up
    vol_up = np.clip(audio_data * 1.5, -1.0, 1.0)
    augmentations.append(('vol_up', vol_up))
    
    # 8. Volume down
    vol_down = audio_data * 0.5
    augmentations.append(('vol_down', vol_down))
    
    # 9. Pitch up + Speed up
    pitch_speed_up = librosa.effects.time_stretch(y=pitch_up, rate=1.1)
    augmentations.append(('pitch_speed_up', pitch_speed_up))
    
    # 10. Pitch down + Speed down
    pitch_speed_down = librosa.effects.time_stretch(y=pitch_down, rate=0.9)
    augmentations.append(('pitch_speed_down', pitch_speed_down))
    
    return augmentations

print("✅ Augmentation pipeline ready.")

✅ Augmentation pipeline ready.


In [4]:
# Recording parameters
SAMPLE_RATE = 22050  
MIC_DEVICE_ID = 1    
recorded_audio = None

# Base Directory explicitly defined here just in case
BASE_DIR = os.path.join('data', 'raw')

# UI Elements
sentence_dropdown = widgets.Dropdown(options=SENTENCES, description='Sentence:')
duration_slider = widgets.FloatSlider(value=3.0, min=1.0, max=10.0, step=0.5, description='Duration (s):')
record_btn = widgets.Button(description='🎤 Start Recording', button_style='danger')
save_btn = widgets.Button(description='💾 Save & Augment', button_style='success', disabled=True)
output_area = widgets.Output()

def on_record_clicked(b):
    global recorded_audio
    with output_area:
        clear_output()
        record_btn.description = 'Recording...'
        record_btn.disabled = True
        save_btn.disabled = True
        
        duration = duration_slider.value
        print(f"🔴 Recording for {duration} seconds. Please speak the phrase: '{sentence_dropdown.value}'")
        
        audio = sd.rec(
            int(duration * SAMPLE_RATE), 
            samplerate=SAMPLE_RATE, 
            channels=1, 
            dtype='float32',
            device=MIC_DEVICE_ID 
        )
        sd.wait() 
        
        recorded_audio = audio.flatten()
        
        print("✅ Recording complete! Listen to the preview below:")
        display(Audio(recorded_audio, rate=SAMPLE_RATE))
        
        record_btn.description = '🎤 Re-Record'
        record_btn.disabled = False
        save_btn.disabled = False

def on_save_clicked(b):
    global recorded_audio
    with output_area:
        if recorded_audio is None:
            print("No audio recorded yet!")
            return
            
        sentence = sentence_dropdown.value
        save_dir = os.path.join(BASE_DIR, sentence)
        os.makedirs(save_dir, exist_ok=True)
        
        # 1. Determine the Sentence ID (s01, s02, etc.) based on its index in the list
        sentence_idx = SENTENCES.index(sentence) + 1
        s_label = f"s{sentence_idx:02d}"
        
        # 2. Count existing .wav files in the directory to determine the starting 'sk' number
        existing_files = [f for f in os.listdir(save_dir) if f.endswith('.wav')]
        current_sk_num = len(existing_files) + 1
        
        # Save Original
        orig_filename = os.path.join(save_dir, f"sk{current_sk_num:02d}_{s_label}.wav")
        sf.write(orig_filename, recorded_audio, SAMPLE_RATE)
        print(f"💾 Saved original to: sk{current_sk_num:02d}_{s_label}.wav")
        current_sk_num += 1 # Increment for the next file
        
        # Generate and save Augmentations
        print("⏳ Generating 10 augmented versions...")
        try:
            augmentations = augment_audio(recorded_audio, SAMPLE_RATE)
            
            for aug_name, aug_audio in augmentations:
                aug_filename = os.path.join(save_dir, f"sk{current_sk_num:02d}_{s_label}.wav")
                sf.write(aug_filename, aug_audio, SAMPLE_RATE)
                current_sk_num += 1 # Increment for each augmentation
                
            print(f"🎉 Successfully saved 11 files (Original + 10 augs) for '{sentence}'.")
        except NameError:
            print("❌ Error: The 'augment_audio' function is missing! Make sure you ran Cell 3.")
            
        # Reset UI
        save_btn.disabled = True
        recorded_audio = None
        record_btn.description = '🎤 Start Recording'
        
        # ---------------------------------------------------------
        # THE FIX: Auto-advance the dropdown to the next sentence
        current_idx = sentence_dropdown.index
        next_idx = (current_idx + 1) % len(SENTENCES)  # Wraps around to 0 if at the end
        sentence_dropdown.index = next_idx
        print(f"\n➡️ Auto-advanced to next sentence: '{sentence_dropdown.value}'")
        # ---------------------------------------------------------

# Attach button clicks
record_btn.on_click(on_record_clicked)
save_btn.on_click(on_save_clicked)

# Display UI
display(widgets.VBox([
    widgets.HTML("<h3>🎤 Speech Data Collection Tool</h3>"),
    sentence_dropdown,
    duration_slider,
    widgets.HBox([record_btn, save_btn]),
    output_area
]))